# 03 — VisTA: Bi-Temporal Change Analysis
**Owner: Person 3**

Priority 2 (mandatory): bi-temporal change description / change-VQA. VisTA is chosen because it produces both a text answer AND a spatial change mask, giving strong visual evidence for the agent.

**Do not fine-tune VisTA initially** — get pretrained inference working and evaluated on CDVQA first. Fine-tuning is a stretch goal only if there's spare time after all mandatory pieces work.

## 1. Environment

In [ ]:
# !pip install torch torchvision pillow --quiet
# !git clone <VisTA repo> vista_repo  # or pip install if packaged
import sys, os
sys.path.insert(0, os.path.abspath('..'))


## 2. Load shared configuration

In [ ]:
from src.utils.io_utils import load_config

config = load_config('../configs/config.yaml')
change_cfg = config['models']['change']
cdvqa_cfg = config['datasets']['cdvqa']
change_cfg

## 3. Load CDVQA and inspect a temporal pair
Confirm image pairing, question/answer format, and whether reference change masks are provided (needed for `change_f1` later).

In [ ]:
# from src.preprocessing.dataset_loader import get_dataloader
# cdvqa_loader = get_dataloader(task='change', split='val', dataset='cdvqa', config=config)
# batch = next(iter(cdvqa_loader))
# batch.keys()


## 4. Preprocess an image pair
Use the shared preprocessing pipeline so change/fusion/vqa never diverge in how they read GeoTIFFs.

In [ ]:
from src.preprocessing.geotiff_utils import read_image
from src.preprocessing.normalize import preprocess_pipeline

# img_t1 = read_image('<path to date-1 image>')
# img_t2 = read_image('<path to date-2 image>')
# arr_t1 = preprocess_pipeline(img_t1.array, config, dataset='cdvqa', model='change')
# arr_t2 = preprocess_pipeline(img_t2.array, config, dataset='cdvqa', model='change')


## 5. Load pretrained VisTA

In [ ]:
# vista = load_vista_pretrained(change_cfg['checkpoint'])
# vista.eval()


## 6. Run pretrained inference — change VQA / description

In [ ]:
# query = 'What changed between these two dates, and where did the change occur?'
# answer, confidence = vista.infer(arr_t1, arr_t2, query)
# answer, confidence


## 7. Generate the change mask
This is the visual evidence the agent will overlay on the image.

In [ ]:
# change_mask = vista.get_change_mask(arr_t1, arr_t2)
# import matplotlib.pyplot as plt
# plt.imshow(change_mask, cmap='hot')
# plt.title('Predicted change mask')
# plt.show()


## 8. Evaluate on CDVQA
Use `src/evaluation/metrics.py::vqa_accuracy` for change-VQA answers and `change_f1` for masks, if CDVQA provides reference masks.

In [ ]:
from src.evaluation import metrics as M

# preds, golds = [], []  # collect answers from eval loop
# print('Change-VQA accuracy:', M.vqa_accuracy(preds, golds))
# pred_masks, gold_masks = [], []  # only if CDVQA has reference masks
# print('Change mask F1:', M.change_f1(pred_masks, gold_masks))


## 9. (Stretch goal only) Fine-tune VisTA on CDVQA
Only attempt this if Sections 1-8 work reliably and there is spare time. Mandatory requirements are already satisfied by pretrained inference above.

In [ ]:
# TODO(Person 3, stretch goal): fine-tuning loop, see
# src/training/train_change.py for the reusable skeleton.


## 10. Export the inference function
Move the stable implementation into `src/models/change_model.py` following the shared `RSModelResult` contract.

In [ ]:
from src.common.schemas import RSModelResult, ResultMetadata, Evidence

# def predict(image_t1, image_t2, query):
#     answer, confidence = ...
#     mask = ...  # np.ndarray or None
#     return RSModelResult(
#         task='change', answer=answer, confidence=confidence,
#         evidence=Evidence(mask=mask),
#         metadata=ResultMetadata(model='change_v1',
#                                  checkpoint=change_cfg['checkpoint'],
#                                  dataset='CDVQA',
#                                  backbone='VisTA'),
#     ).to_dict()
